# ROMS Hawaiʻi reanalysis v2.0: scratch store on Source Cooperative

Step 2 of the reanalysis store ([issue #11](https://github.com/ocean-icechunks/pacioos/issues/11)),
after the local smoke test (`reanalysis-smoke-test-local.ipynb`). It changes one thing:
**where the store is written**. The same four-day store (2009-12-30 to 2010-01-02, a write and
an append) goes to a **scratch prefix** on Source Cooperative, laid out like production:

| | Scratch (this notebook) | Production (later) |
|---|---|---|
| store | `ocean-icechunks/test-repo/pacioos/roms_hiig_reanalysis/icechunk` | `ocean-icechunks/pacioos/roms_hiig_reanalysis/icechunk` |
| viewer | `.../test-repo/pacioos/roms_hiig_reanalysis/viewer/` | `.../pacioos/roms_hiig_reanalysis/viewer/` |
| README | `.../test-repo/pacioos/roms_hiig_reanalysis/README.md` | the project README |

Then it reads the store back **anonymously from its public URL**, as any reader would, and
checks the two hosts a browser needs.

## Running it

It needs Source Cooperative write credentials for the `ocean-icechunks` account
(`source-coop login --duration 1d --port 8400`; see `upload/README.md` for a JupyterHub). It
**writes only under `test-repo/`**, and refuses any other prefix. If the scratch store already
holds data it skips the write and checks what is there; set `RESET_SCRATCH = True` and run the
guarded cell at the end to start again.

The viewer is built and uploaded from the shell with `../publish_viewer.py` (section 7).

In [1]:
# Dependencies. In the git repo: pip install -r requirements.txt  (from this folder)
# icechunk 2.x requires Python >= 3.12.

In [2]:
import sys
# pacioos_virtual.py, icechunk_utils.py and publish_viewer.py are one level up, in stores/.
sys.path.insert(0, '..')

import time
import urllib.request

import icechunk
import netCDF4
import numpy as np
import pandas as pd
import virtualizarr
import xarray as xr
import zarr

import pacioos_virtual as pv
import reanalysis_store as rs

for name, mod in [('icechunk', icechunk), ('virtualizarr', virtualizarr), ('xarray', xr),
                  ('zarr', zarr), ('netCDF4', netCDF4)]:
    print(f'{name:13s}{mod.__version__}')
print(f'{"python":13s}{sys.version.split()[0]}')

icechunk     2.3.1
virtualizarr 2.8.0
xarray       2026.9.0
zarr         3.4.2
netCDF4      1.7.5.1.post1
python       3.12.12


## 1. Configuration

In [3]:
SC_BUCKET = 'ocean-icechunks'
SCRATCH = 'test-repo/pacioos/roms_hiig_reanalysis'      # source.coop/ocean-icechunks/test-repo
STORE_PREFIX = f'{SCRATCH}/icechunk'
PUBLIC = 'https://data.source.coop'
STORE_URL = f'{PUBLIC}/{SC_BUCKET}/{STORE_PREFIX}'
VIEWER_URL = f'{PUBLIC}/{SC_BUCKET}/{SCRATCH}/viewer/index.html'

# Never anything but scratch from this notebook.
assert STORE_PREFIX.startswith('test-repo/') and '/pacioos/' in STORE_PREFIX

RUN_WRITE = True          # False: only check the scratch store that is already there
RESET_SCRATCH = False     # True: allow the guarded delete cell at the end to run

SMALL_DAYS = pd.date_range('2009-12-30', '2010-01-02', freq='D')   # crosses a year folder
zarr.config.set({'async.concurrency': 128})
UA = {'User-Agent': 'pacioos-reanalysis-scratch'}   # the edge 403s Python-urllib's default
print('store:', STORE_URL)

store: https://data.source.coop/ocean-icechunks/test-repo/pacioos/roms_hiig_reanalysis/icechunk


## 2. Build the virtual dataset

Exactly as in the local smoke test: four files scanned, references to one depth plane per
chunk, metadata from the saved ERDDAP copy.

In [4]:
main_attrs, ssh_attrs, fetched = rs.load_metadata()
scans = [pv.scan_h5(rs.key(d)) for d in SMALL_DAYS]
VAR_ATTRS = rs.variable_attrs(main_attrs, ssh_attrs, scans[0]['vars']['mask']['attrs'])

first = rs.build_dataset(scans[:2], VAR_ATTRS, main_attrs)
rest = rs.build_dataset(scans[2:], VAR_ATTRS, main_attrs).drop_vars(['depth', 'lat', 'lon', 'mask'])
# An append rewrites the root attributes: give the appended part the whole span's.
rest.attrs = rs.global_attrs(main_attrs, pd.DatetimeIndex(
    np.concatenate([first.time.values, rest.time.values])))
print(f'metadata saved {fetched}; {first.sizes["time"]} + {rest.sizes["time"]} steps')

metadata saved 2026-11-14T00:19:42Z; 16 + 16 steps


## 3. Write to the scratch prefix

`open_source_icechunk_repo` (`../icechunk_utils.py`) refreshes the Source Cooperative token
through the `source-coop` CLI and refuses to start with less than 15 minutes left on it. The
repository config is the production one (`rs.repository_config()`): one virtual chunk
container for `https://data.source.coop/ocean-icechunks/pacioos/`, saved into the repository
so readers find it, and manifests split every 2,920 steps.

In [5]:
from icechunk_utils import open_source_icechunk_repo

if RUN_WRITE:
    repo, storage, _creds, time_left = open_source_icechunk_repo(
        bucket=SC_BUCKET, prefix=STORE_PREFIX, config=rs.repository_config())
    if repo is None:
        raise SystemExit('Stopped: Source Cooperative token too close to expiry.')
    repo = repo.reopen(authorize_virtual_chunk_access=pv.reader_authorization())

    history = list(repo.ancestry(branch='main'))
    if len(history) > 1:
        print(f'{STORE_PREFIX} already holds {len(history) - 1} commit(s); not writing. '
              'Set RESET_SCRATCH and run the delete cell at the end to start again.')
    else:
        repo.save_config()
        t0 = time.perf_counter()
        session = repo.writable_session('main')
        first.vz.to_icechunk(session.store)
        session.commit(f'{SMALL_DAYS[0]:%Y-%m-%d}..{SMALL_DAYS[1]:%Y-%m-%d}')
        session = repo.writable_session('main')       # a committed session is read-only
        rest.vz.to_icechunk(session.store, append_dim='time')
        snapshot = session.commit(f'append {SMALL_DAYS[2]:%Y-%m-%d}..{SMALL_DAYS[3]:%Y-%m-%d}')
        repo.create_tag('v2.0', snapshot_id=snapshot)
        print(f'written, appended and tagged v2.0 in {time.perf_counter() - t0:.1f}s')
    for entry in repo.ancestry(branch='main'):
        print(' ', entry.written_at.strftime('%Y-%m-%d %H:%M:%S'), entry.message)
else:
    print('RUN_WRITE is False: nothing written.')

Created new Icechunk repo
Time remaining on token: 11:51:22.958314


written, appended and tagged v2.0 in 3.5s
  2026-11-14 00:27:08 append 2010-01-01..2010-01-02
  2026-11-14 00:27:06 2009-12-30..2009-12-31
  2026-11-14 00:27:04 Repository initialized


## 4. Read it back anonymously from the public URL

Nothing from the writer is reused: an `http_storage` on the public URL, no credentials, the
virtual chunk container read from the saved config and authorized for anonymous HTTP
(`rs.open_store`). This is the reader's path.

In [6]:
t0 = time.perf_counter()
reader = icechunk.Repository.open(icechunk.http_storage(STORE_URL))
containers = list(reader.config.virtual_chunk_containers or [])
print('tags:', list(reader.list_tags()), '| containers:', containers)
ds = rs.open_store(icechunk.http_storage(STORE_URL), ref='v2.0')
raw = rs.open_store(icechunk.http_storage(STORE_URL), ref='v2.0', mask_and_scale=False)
print(f'opened in {time.perf_counter() - t0:.2f}s\n')

assert containers == [rs.URL_PREFIX]
assert dict(ds.sizes) == {'time': 32, 'depth': 36, 'lat': 195, 'lon': 295}
assert pd.DatetimeIndex(ds.time.values).isin(rs.STEPS).all()
assert ds.temp.encoding['chunks'] == (1, 1, 195, 295)
assert ds.temp.attrs['units'] == 'Celsius' and ds.attrs['title'] == rs.TITLE
assert (ds.attrs['time_coverage_start'], ds.attrs['time_coverage_end']) == (
    '2009-12-30T00:00:00Z', '2010-01-02T21:00:00Z')
ds

tags: ['v2.0'] | containers: ['https://data.source.coop/ocean-icechunks/pacioos/']


opened in 1.65s



<xarray.Dataset> Size: 1GB
Dimensions:  (time: 32, depth: 36, lat: 195, lon: 295)
Coordinates:
  * time     (time) datetime64[ns] 256B 2009-12-30 ... 2010-01-02T21:00:00
  * depth    (depth) float32 144B 0.25 1.0 2.0 5.0 ... 4.5e+03 5e+03 5.5e+03
  * lat      (lat) float32 780B 17.02 17.05 17.09 17.13 ... 23.91 23.95 23.98
  * lon      (lon) float32 1kB -163.8 -163.8 -163.8 ... -152.6 -152.6 -152.5
Data variables:
    v        (time, depth, lat, lon) float32 265MB ...
    mask     (lat, lon) int16 115kB ...
    salt     (time, depth, lat, lon) float32 265MB ...
    u        (time, depth, lat, lon) float32 265MB ...
    zeta     (time, lat, lon) float32 7MB ...
    temp     (time, depth, lat, lon) float32 265MB ...
Attributes: (12/54)
    acknowledgement:            The Pacific Islands Ocean Observing System (P...
    comment:                    Model runs produced by Brian Powell (powellb@...
    Conventions:                CF-1.6, ACDD-1.3
    creator_email:              powellb@hawaii.edu
    creator_institution:        University of Hawaii at Manoa
    creator_name:               Brian S. Powell
    ...                         ...
    summary:                    Regional Ocean Modeling System (ROMS) 3-hourl...
    time_coverage_resolution:   PT3H
    title:                      Regional Ocean Modeling System (ROMS): Main H...
    time_coverage_start:        2009-12-30T00:00:00Z
    time_coverage_end:          2010-01-02T21:00:00Z
    source_files:               https://data.source.coop/ocean-icechunks/paci...

In [7]:
# Values against netCDF-C, first and last step of each file, surface and deepest level.
t0 = time.perf_counter()
checked, bad = 0, []
for f, day in enumerate(SMALL_DAYS):
    with netCDF4.Dataset(rs.URL_PREFIX + rs.key(day) + '#mode=bytes') as nc:
        for name in rs.SCIENCE:
            v = nc[name]
            v.set_auto_maskandscale(False)
            for lead in (0, 7):
                for k in ((0, 35) if name != 'zeta' else (None,)):
                    exp = v[lead] if k is None else v[lead, k]
                    sel = dict(time=8 * f + lead) | ({} if k is None else dict(depth=k))
                    checked += 1
                    if not np.array_equal(raw[name].isel(sel).values, exp):
                        bad.append((day.date(), name, lead, k))
print(f'{checked} planes compared with netCDF-C in {time.perf_counter() - t0:.0f}s, '
      f'mismatches: {bad}')
assert not bad

surface = ds.temp.isel(time=0, depth=0).values
assert int(np.isnan(surface).sum()) == int((ds.mask.values == 0).sum()) == 1118
print(f'surface land cells read as NaN: 1118; temp {np.nanmin(surface):.2f} .. '
      f'{np.nanmax(surface):.2f} Celsius')

t0 = time.perf_counter()
ds.temp.isel(time=31, depth=0).load()
print(f'one surface map from the public store: {time.perf_counter() - t0:.2f}s')

72 planes compared with netCDF-C in 41s, mismatches: []


surface land cells read as NaN: 1118; temp 23.51 .. 26.51 Celsius
one surface map from the public store: 0.14s


## 5. The two hosts a browser needs

A virtual store needs CORS on two hosts: the one serving the store (metadata, coordinates) and
the one serving the referenced bytes. Here both are `data.source.coop`, but they are checked
separately, with a ranged GET carrying an `Origin` header, because that is the request a
browser makes. `Access-Control-Allow-Origin` on that response is what decides.

In [8]:
import boto3
from icechunk_utils import get_source_credentials

creds, _exp = get_source_credentials()
s3 = boto3.client('s3', endpoint_url=creds['endpoint_url'], region_name=creds['region_name'],
                  aws_access_key_id=creds['aws_access_key_id'],
                  aws_secret_access_key=creds['aws_secret_access_key'],
                  aws_session_token=creds['aws_session_token'])
store_keys = [o['Key'] for p in s3.get_paginator('list_objects_v2').paginate(
    Bucket=SC_BUCKET, Prefix=STORE_PREFIX + '/') for o in p.get('Contents', [])]
print(f'{len(store_keys)} objects in the scratch store, '
      f'{sum(o["Size"] for p in s3.get_paginator("list_objects_v2").paginate(Bucket=SC_BUCKET, Prefix=STORE_PREFIX + "/") for o in p.get("Contents", [])) / 1e3:.0f} kB')

manifest = next(k for k in store_keys if '/manifests/' in k)
for label, url in [('store (a manifest)', f'{PUBLIC}/{SC_BUCKET}/{manifest}'),
                   ('referenced NetCDF', rs.URL_PREFIX + rs.key(SMALL_DAYS[0]))]:
    req = urllib.request.Request(url, headers={**UA, 'Range': 'bytes=0-15',
                                               'Origin': 'https://example.org'})
    with urllib.request.urlopen(req) as r:
        acao = r.headers.get('Access-Control-Allow-Origin')
        print(f'{label:20s} {r.status}  Access-Control-Allow-Origin: {acao}')
        assert r.status == 206 and acao == '*'

29 objects in the scratch store, 110 kB
store (a manifest)   206  Access-Control-Allow-Origin: *
referenced NetCDF    206  Access-Control-Allow-Origin: *


## 6. The scratch README

Three things and no more: that this is a scratch copy, the viewer link, and the code to load
the store. The loading code was checked by downloading the published README and running its
code block exactly as written, in a clean Python 3.12 venv holding only the packages its install
line names (2026-11-14: it printed the dataset and a mean surface temperature of 25.26).

In [9]:
README = f'''# SCRATCH: ROMS Hawaiʻi reanalysis v2.0, virtual Icechunk test store

**This is a scratch copy for testing. It may be rewritten or deleted at any time.** It holds
four days (2009-12-30 to 2010-01-02) of the PacIOOS ROMS reanalysis for the main Hawaiian
Islands, as references into the NetCDF copies at
`{rs.URL_PREFIX}roms_hiig_reanalysis/netcdf/v2.0/`. The real store will be at
`{PUBLIC}/{SC_BUCKET}/pacioos/roms_hiig_reanalysis/icechunk`.
Code and notes: https://github.com/ocean-icechunks/pacioos (issue #11).

## View it in a browser

{VIEWER_URL}

## Load it in Python

```python
# pip install "icechunk>=2.3" xarray zarr      (Python 3.12 or newer)
import icechunk
import xarray as xr

url = "{STORE_URL}"
repo = icechunk.Repository.open(icechunk.http_storage(url))
# The data stay in the NetCDF files, outside the store: allow anonymous reads of them.
auth = {{p: icechunk.credentials.HttpAccess for p in repo.config.virtual_chunk_containers or []}}
repo = repo.reopen(authorize_virtual_chunk_access=auth)
ds = xr.open_zarr(repo.readonly_session(tag="v2.0").store, consolidated=False, chunks=None)
print(ds)
print(float(ds.temp.isel(time=0, depth=0).mean()))
```
'''
print(README)

# SCRATCH: ROMS Hawaiʻi reanalysis v2.0, virtual Icechunk test store

**This is a scratch copy for testing. It may be rewritten or deleted at any time.** It holds
four days (2009-12-30 to 2010-01-02) of the PacIOOS ROMS reanalysis for the main Hawaiian
Islands, as references into the NetCDF copies at
`https://data.source.coop/ocean-icechunks/pacioos/roms_hiig_reanalysis/netcdf/v2.0/`. The real store will be at
`https://data.source.coop/ocean-icechunks/pacioos/roms_hiig_reanalysis/icechunk`.
Code and notes: https://github.com/ocean-icechunks/pacioos (issue #11).

## View it in a browser

https://data.source.coop/ocean-icechunks/test-repo/pacioos/roms_hiig_reanalysis/viewer/index.html

## Load it in Python

```python
# pip install "icechunk>=2.3" xarray zarr      (Python 3.12 or newer)
import icechunk
import xarray as xr

url = "https://data.source.coop/ocean-icechunks/test-repo/pacioos/roms_hiig_reanalysis/icechunk"
repo = icechunk.Repository.open(icechunk.http_storage(url))
# The data 

In [10]:
s3.put_object(Bucket=SC_BUCKET, Key=f'{SCRATCH}/README.md', Body=README.encode(),
              ContentType='text/markdown; charset=utf-8')
req = urllib.request.Request(f'{PUBLIC}/{SC_BUCKET}/{SCRATCH}/README.md', headers=UA)
with urllib.request.urlopen(req) as r:
    assert r.read().decode() == README
    print(r.status, r.headers['Content-Type'], f'{PUBLIC}/{SC_BUCKET}/{SCRATCH}/README.md')

200 text/markdown; charset=utf-8 https://data.source.coop/ocean-icechunks/test-repo/pacioos/roms_hiig_reanalysis/README.md


## 7. The viewer

Built from gridlook `main` and uploaded from the shell, from `stores/`:

```bash
git clone https://github.com/eeholmes/gridlook ~/gridlook && (cd ~/gridlook && npm ci)
python publish_viewer.py --product roms_hiig_reanalysis-scratch --build ~/gridlook
```

This cell only checks that it is there. Whether it draws is for a person with a browser to
say; a Python read does not show it.

In [11]:
req = urllib.request.Request(VIEWER_URL, method='HEAD', headers=UA)
with urllib.request.urlopen(req) as r:
    print(r.status, r.headers['Content-Type'])
    assert r.status == 200 and r.headers['Content-Type'].startswith('text/html')
print(f'{VIEWER_URL}#icechunk+{STORE_URL}::varname=temp')

200 text/html; charset=utf-8
https://data.source.coop/ocean-icechunks/test-repo/pacioos/roms_hiig_reanalysis/viewer/index.html#icechunk+https://data.source.coop/ocean-icechunks/test-repo/pacioos/roms_hiig_reanalysis/icechunk::varname=temp


## Troubleshooting: delete the scratch store

Guarded with `%%script false --no-raise`, so running all cells never runs it. To use it, set
`RESET_SCRATCH = True` above, remove the first line of the cell, and run it. It lists what it
would delete before deleting, gathers every key before deleting any, and names the exact
prefix inline. It cannot reach anything outside `test-repo/pacioos/roms_hiig_reanalysis/icechunk/`.

In [12]:
%%script false --no-raise
assert RESET_SCRATCH
PREFIX = 'test-repo/pacioos/roms_hiig_reanalysis/icechunk/'
keys = [o['Key'] for p in s3.get_paginator('list_objects_v2').paginate(
    Bucket='ocean-icechunks', Prefix=PREFIX) for o in p.get('Contents', [])]
print(f'{len(keys)} objects under ocean-icechunks/{PREFIX}')
for k in keys[:10]:
    print('  ', k)
for i in range(0, len(keys), 1000):
    s3.delete_objects(Bucket='ocean-icechunks',
                      Delete={'Objects': [{'Key': k} for k in keys[i:i + 1000]]})
print('deleted', len(keys))